In [ ]:
from common import *
from db import get_engine

engine = get_engine()

## 1. Ümumi KPI nəticələrinin hazırlanması

Bu mərhələdə Silver cədvəlindən əsas uçuş KPI-ları hesablanır. Məqsəd Power BI və reporting üçün hazır yekun nəticə yaratmaqdır.

In [ ]:
gold_kpi = pd.read_sql("""
SELECT
    COUNT(*) AS TOTAL_FLIGHTS,
    SUM(CASE WHEN FLIGHT_STATUS = 'Completed' THEN 1 ELSE 0 END) AS COMPLETED_FLIGHTS,
    SUM(CANCELLED) AS CANCELLED_FLIGHTS,
    SUM(DIVERTED) AS DIVERTED_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
""", con=engine)

gold_kpi

#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış ümumi KPI nəticələri `gold.flight_kpi_summary` cədvəlinə yazılır.

In [ ]:
gold_kpi.to_sql(
    "flight_kpi_summary",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

## 2. Aviaşirkətlər üzrə performans nəticələrinin hazırlanması

Bu mərhələdə aviaşirkətlər üzrə ümumi uçuş sayı, gecikmə faizi və orta çatma gecikməsi hesablanır. Məqsəd nəticəni Power BI üçün Gold layer-da hazırlamaqdır.

In [ ]:
gold_airline = pd.read_sql("""
SELECT
    AIRLINE_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY AIRLINE_NAME
ORDER BY DELAY_RATE
""", con=engine)

gold_airline

#### Nəticənin Gold cədvəlinə yazılması

Aşağıdakı addımda hazırlanmış aviaşirkət performans nəticələri `gold.airline_performance` cədvəlinə yazılır.

In [ ]:
gold_airline.to_sql(
    "airline_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

## 3. Aylıq performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların aylar üzrə ümumi sayı, gecikmə faizi, orta çatma gecikməsi, ləğv və diversion faizləri hesablanır.
Məqsəd nəticələri Power BI üçün hazır Gold cədvəlinə çevirməkdir.

In [ ]:
gold_monthly = pd.read_sql("""
SELECT
    MONTH,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY,
    ROUND(100.0 * SUM(CANCELLED) / COUNT(*), 2) AS CANCELLATION_RATE,
    ROUND(100.0 * SUM(DIVERTED) / COUNT(*), 2) AS DIVERSION_RATE
FROM silver.flights_clean
GROUP BY MONTH
""", con=engine)

gold_monthly

#### Nəticənin Gold cədvəlinə yazılması

Hazırlanmış aylıq performans nəticələri `gold.monthly_performance` cədvəlinə yazılır.

In [ ]:
gold_monthly.to_sql(
    "monthly_performance",
    con=engine,
    schema="gold",
    if_exists="replace",
    index=False
)

## 4. Günün vaxtına görə performans nəticələrinin hazırlanması

Bu mərhələdə uçuşların günün müxtəlif vaxtlarına görə ümumi sayı, gecikmə faizi və orta çatma gecikməsi hesablanır.
Məqsəd hansı vaxt aralığında uçuş performansının daha zəif olduğunu Power BI-da göstərməkdir.

In [ ]:
gold_departure_period = pd.read_sql("""
SELECT
    DEPARTURE_PERIOD,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE,
    ROUND(AVG(ARRIVAL_DELAY), 2) AS AVG_ARRIVAL_DELAY
FROM silver.flights_clean
GROUP BY DEPARTURE_PERIOD
ORDER BY DELAY_RATE DESC
""", con=engine)

gold_departure_period

## 5. Marşrutlar üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin və destination hava limanları üzrə uçuş sayı və gecikmə faizi hesablanır.
Məqsəd gecikmənin daha yüksək olduğu marşrutları Power BI üçün hazırlamaqdır.

In [ ]:
gold_route = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    DESTINATION_AIRPORT,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(100.0 * SUM(IS_DELAYED) / COUNT(IS_DELAYED), 2) AS DELAY_RATE
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
HAVING COUNT(*) >= 500
""", con=engine)

gold_route

## 6. Hava limanları üzrə performans nəticələrinin hazırlanması

Bu mərhələdə origin hava limanları üzrə orta taxi-out müddəti hesablanır.
Məqsəd yerüstü əməliyyatlarda daha çox vaxt itirilən hava limanlarını Power BI üçün hazırlamaqdır.

In [ ]:
gold_airport = pd.read_sql("""
SELECT
    ORIGIN_AIRPORT,
    ORIGIN_AIRPORT_NAME,
    COUNT(*) AS TOTAL_FLIGHTS,
    ROUND(AVG(TAXI_OUT), 2) AS AVG_TAXI_OUT
FROM silver.flights_clean
GROUP BY ORIGIN_AIRPORT, ORIGIN_AIRPORT_NAME
""", con=engine)

gold_airport